# 00｜課程導覽

用小型資料練習辨認樣本、特徵與目標，並觀察訓練／測試切分。

從上到下執行程式格。第一格安裝缺少的套件，第二格準備繪圖設定。


In [ ]:
# 第 1 格：安裝本 Notebook 實際使用的套件。
import importlib.util, subprocess, sys
_needed = {'numpy': 'numpy>=1.26,<3', 'pandas': 'pandas>=2.0,<3', 'matplotlib': 'matplotlib>=3.7,<4', 'sklearn': 'scikit-learn>=1.4,<2'}
_missing = [spec for module, spec in _needed.items() if importlib.util.find_spec(module) is None]
if _missing:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *_missing])


In [ ]:
# 第 2 格：本 Notebook 的輸出位置與繪圖設定。
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

SEED = 42
ROOT = Path.cwd()
OUT = ROOT / ("outputs" if (ROOT / "pyproject.toml").exists() else "mlcourse_outputs")
plt.rcParams.update({"figure.figsize": (8, 4.5), "figure.dpi": 110,
                     "axes.grid": True, "grid.alpha": 0.2, "font.size": 11})

def savefig(name):
    folder = OUT / "figures"
    folder.mkdir(parents=True, exist_ok=True)
    plt.gcf().savefig(folder / f"{name}.png", dpi=150, bbox_inches="tight")
    plt.gcf().savefig(folder / f"{name}.svg", bbox_inches="tight")
    plt.show()


## 1. 樣本、特徵與目標

每一列是一筆學生的練習紀錄；練習時數與次數是特徵，是否通過是目標。

In [ ]:
samples = pd.DataFrame({
    "study_hours": [1, 2, 2, 3, 3, 4, 4, 5, 5, 6, 6, 7],
    "practice_count": [0, 0, 1, 1, 2, 1, 3, 2, 4, 3, 5, 4],
    "passed": [0, 0, 0, 0, 1, 0, 1, 1, 1, 1, 1, 1],
})
X = samples[["study_hours", "practice_count"]]
y = samples["passed"]
print("樣本數:", len(samples), "特徵數:", X.shape[1], "目標類別:", sorted(y.unique()))
print(samples)


## 2. 先切分，再評估

固定隨機種子可重現這次切分；測試列應留到模型與參數確定後才使用。

In [ ]:
from sklearn.model_selection import train_test_split
train_ids, test_ids = train_test_split(samples.index.to_numpy(), test_size=0.25,
                                      random_state=SEED, stratify=y)
assert not set(train_ids) & set(test_ids)
print("訓練列:", sorted(train_ids), "測試列:", sorted(test_ids))
for ids, label, marker in [(train_ids, "train", "o"), (test_ids, "test", "x")]:
    plt.scatter(X.loc[ids, "study_hours"], X.loc[ids, "practice_count"],
                label=label, marker=marker, s=80)
plt.xlabel("study hours"); plt.ylabel("practice count"); plt.legend()
savefig("00_samples_and_split")


## 練習

1. 指出表格中一筆樣本、兩個特徵與一個目標。
2. 改變 `SEED` 後重跑切分，觀察測試列是否改變。
3. 討論為何不能看過測試結果後再選模型。